# ITD-X reference + pseudo-labels (Free T4)

Uses IIT-Roorkee's `best_xl_ITD_v1.2.pt` (114MB, CUDA-only — never an edge candidate) two ways:
**A. Compare** — ITD-X vs finale vs candidate on TrafficCAM val (head-to-head table).
**B. Pseudo-label** — ITD-X drafts YOLO labels for unlabelled TrafficCAM frames (model-derived GT, reported separately from human GT, spot-checked before trust).

| | |
|---|---|---
| Upload | `best_xl_ITD_v1.2.pt` to `/content` (Files pane) |
| Class map | ITD-X 8 → contract 6: two wheeler→motorcycle, autorickshaw→auto, car→car, bus→bus, LCV→truck, truck→truck, bicycle→bicycle, pedestrain→dropped |
| Hand-back | `itd_x_<date>.zip` (compare table + pseudo labels + overlay grid) → `notebooks/training_output_zips/` |

In [ ]:
# Cell 0 — GPU check + setup + branch-pinned repo (proves which code runs)
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU, then re-run'
!pip install -q ultralytics opencv-python-headless gdown
!apt-get install -y -qq git-lfs > /dev/null 2>&1; git lfs install --skip-repo > /dev/null 2>&1
import os, shutil, subprocess, sys
REPO = '/content/SGP-IV'
BRANCH = 'feat/policy-ports-screenshot-green'
def _git(*a):
    return subprocess.run(['git', '-C', REPO, *a], capture_output=True, text=True)
_cur = _git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip() \
    if os.path.isdir(REPO + '/.git') else ''
if _cur != BRANCH:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/Bobbymkr/SGP-IV.git', REPO], check=True)
print('repo:', _git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip(),
      _git('rev-parse', '--short', 'HEAD').stdout.strip())
assert os.path.exists('/content/best_xl_ITD_v1.2.pt'), 'upload best_xl_ITD_v1.2.pt first'
print('setup ok')

In [ ]:
# Cell 1 — data (idempotent; same converter as the training loop)
import os
if not os.path.exists('/content/tcam/data.yaml'):
    !mkdir -p /content/trafficcam
    !test -f /content/trafficcam/Fully_annotate.zip || gdown https://drive.google.com/uc?id=1h4oUqECDF05vSYMkYgz0aUc_RRzHlh3e -O /content/trafficcam/Fully_annotate.zip
    !unzip -q -o /content/trafficcam/Fully_annotate.zip -d /content/trafficcam
    !cd /content/SGP-IV && python scripts/prepare_dataset.py /content/trafficcam --out /content/tcam --source trafficcam --option B
    !cd /content/SGP-IV && python scripts/prepare_dataset.py /content/tcam --check
    assert os.path.exists('/content/tcam/data.yaml'), 'STOP: converter failed - read output above'
else:
    print('tcam exists, skipped')

In [ ]:
# Cell 2 — A. Compare, taxonomy-aligned AND self-verified.
# (Lesson: ultralytics resolves symlinks, so EV images must be COPIES -
#  a symlink silently re-points label lookup at the source labels.)
import glob, os, shutil
from collections import Counter
CONTRACT2ITD = {0: 2, 1: 0, 2: 3, 3: 5, 4: 6, 5: 1}  # car,moto,bus,truck,bicycle,auto
ITD_NAMES = ['two wheeler', 'autorickshaw', 'car', 'bus', 'LCV', 'truck',
             'bicycle', 'pedestrain']
EV = '/content/tcam_eval_itdx'
shutil.rmtree(EV, ignore_errors=True)  # no stale-label reuse, ever
for split in ('train', 'val'):
    os.makedirs(f'{EV}/labels/{split}')
    for lp in glob.glob(f'/content/tcam/labels/{split}/*.txt'):
        out = []
        for ln in open(lp):
            c, *box = ln.split()
            out.append(f"{CONTRACT2ITD[int(c)]} {' '.join(box)}")
        open(f"{EV}/labels/{split}/{os.path.basename(lp)}", 'w').write('\n'.join(out) + '\n')
    shutil.copytree(f'/content/tcam/images/{split}', f'{EV}/images/{split}')
def _counts(pat):
    c = Counter()
    for lp in glob.glob(pat):
        for ln in open(lp):
            c[int(ln.split()[0])] += 1
    return c
_src = _counts('/content/tcam/labels/val/*.txt')
_dst = _counts(f'{EV}/labels/val/*.txt')
_ni = len(glob.glob(f'{EV}/images/val/*.jpg'))
print('src(contract) idx->n:', dict(sorted(_src.items())))
print('dst(ITD)      idx->n:', dict(sorted(_dst.items())), '| ev images:', _ni)
assert sum(_src.values()) == sum(_dst.values()) > 0, 'STOP: label copy broken'
assert _src != _dst, 'STOP: remap did not take - dst identical to src'
assert _ni == 180, f'STOP: ev images missing ({_ni})'
with open(f'{EV}/data.yaml', 'w') as f:
    f.write(f'path: {EV}\ntrain: images/train\nval: images/val\ntest: images/val\nnames:\n')
    for i, n in enumerate(ITD_NAMES):
        f.write(f"  {i}: '{n}'\n")
print(open(f'{EV}/data.yaml').read())
!yolo val model=/content/best_xl_ITD_v1.2.pt data=/content/tcam_eval_itdx/data.yaml split=val 2>&1 | tail -12
print('PASTE-BACK: overall mAP50 + per-class rows above (verified aligned)')


In [ ]:
# Cell 3 — B. Pseudo-label: ITD-X drafts contract labels for unlabelled frames
# Source: every 6th unlabelled frame of each Fully_annotate video (frame6..58 step 6
# have no JSON). Capped for time; raise CAP for a bigger pool.
import glob, os
from ultralytics import YOLO
CAP, CONF = 300, 0.35
ITD2CONTRACT = {0: 1, 1: 5, 2: 0, 3: 2, 4: 3, 5: 3, 6: 4, 7: None}  # 7=pedestrain: drop
model = YOLO('/content/best_xl_ITD_v1.2.pt')
OUT = '/content/pseudo_itd_x'
os.makedirs(OUT + '/images', exist_ok=True)
os.makedirs(OUT + '/labels', exist_ok=True)
pool = []
for v in sorted(glob.glob('/content/trafficcam/Fully_annotate/*/')):
    pool.extend(sorted(glob.glob(v + 'frame*.jpg'))[1::3])  # skip frame0 (human-labelled)
pool = pool[:CAP]
print('pool:', len(pool))
n_kept = 0
for p in pool:
    r = model(p, conf=CONF, verbose=False)[0]
    lines = []
    for b, c in zip(r.boxes.xywhn.tolist(), r.boxes.cls.tolist()):
        m = ITD2CONTRACT.get(int(c))
        if m is None:
            continue
        cx, cy, w, h = b
        lines.append(f'{m} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    if lines:
        import shutil
        vdir = os.path.basename(os.path.dirname(p))
        stem = f'{vdir}_{os.path.splitext(os.path.basename(p))[0]}'
        shutil.copy(p, f'{OUT}/images/{stem}.jpg')
        open(f'{OUT}/labels/{stem}.txt', 'w').write('\n'.join(lines) + '\n')
        n_kept += 1
print(f'pseudo-labelled frames: {n_kept}/{len(pool)} (empty frames skipped)')

In [ ]:
# Cell 4 — spot-check grid (human eyes before trust: 12 labelled overlays)
import glob, random
import cv2
random.seed(42)
ims = sorted(glob.glob('/content/pseudo_itd_x/images/*.jpg'))
NAMES = ['car', 'motorcycle', 'bus', 'truck', 'bicycle', 'auto']
pick = random.sample(ims, min(12, len(ims)))
pick = pick[: len(pick) // 4 * 4]  # full rows only
assert pick, 'STOP: too few pseudo-labelled frames for a grid'
tiles = []
for p in pick:
    img = cv2.imread(p)
    h, w = img.shape[:2]
    stem = p.rsplit('/', 1)[-1][:-4]
    for ln in open(f'/content/pseudo_itd_x/labels/{stem}.txt'):
        c, cx, cy, bw, bh = ln.split()
        x1, y1 = int((float(cx) - float(bw) / 2) * w), int((float(cy) - float(bh) / 2) * h)
        x2, y2 = int((float(cx) + float(bw) / 2) * w), int((float(cy) + float(bh) / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, NAMES[int(c)], (x1, max(0, y1 - 5)), 0, 0.7, (0, 255, 0), 2)
    tiles.append(cv2.resize(img, (640, 360)))
grid = cv2.vconcat([cv2.hconcat(tiles[i:i + 4]) for i in range(0, len(tiles), 4)])
cv2.imwrite('/content/spotcheck.jpg', grid)
print('saved /content/spotcheck.jpg - eyeball it: tight boxes, right classes, no crowds merged?')
from google.colab import files
files.download('/content/spotcheck.jpg')

In [ ]:
# Cell 5 — package hand-back (pseudo labels are model-derived GT: labelled as such)
import datetime
stamp = datetime.date.today().isoformat()
open('/content/pseudo_itd_x/SOURCE.txt', 'w').write(
    'pseudo-labels by best_xl_ITD_v1.2.pt (IIT Roorkee, CC-BY-NC, research-only). '
    'Model-derived, NOT human GT. Spot-check /content/spotcheck.jpg before use.\n')
zipf = f'/content/itd_x_{stamp}.zip'
!cd /content && zip -qr {zipf} pseudo_itd_x spotcheck.jpg
!ls -la {zipf}
from google.colab import files
files.download(zipf)
print('HAND-BACK:', zipf)
print('PASTE-BACK: Cell-2 mAP table + pseudo count + spot-check verdict (ship / ship-with-cuts / reject)')